In [1]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack

# 1. CARGAMOS BASES REALES
print("📥 Cargando bases de datos finales...")

# El archivo generado en Segmentación ( 200k leads)
df_leads = pd.read_parquet('../users_with_target_v2_completo.parquet')

# El catálogo de caballos (donde buscaremos las recomendaciones)
df_horses = pd.read_parquet('../horses_listings_limpio.parquet')

# 2. LIMPIEZA RÁPIDA (Aseguramos que los nombres coincidan)
df_horses.columns = [c.lower().strip() for c in df_horses.columns]
df_leads.columns = [c.lower().strip() for c in df_leads.columns]

print(f"✅ Leads cargados: {len(df_leads)}")
print(f"✅ Caballos cargados: {len(df_horses)}")

📥 Cargando bases de datos finales...
✅ Leads cargados: 200000
✅ Caballos cargados: 780


In [3]:
# --- PREPARACIÓN DEL MOTOR ---
# Vectorizamos las características de los caballos
tfidf = TfidfVectorizer(max_features=100)
df_horses['caracteristicas'] = (
    df_horses['breed'].fillna('') + " " + 
    df_horses['color'].fillna('')
).str.lower()
matrix_features = tfidf.fit_transform(df_horses['caracteristicas'])

# Escalamos precios
scaler = MinMaxScaler()
price_scaled = scaler.fit_transform(df_horses[['price']])

# Creamos la matriz final (CSR para evitar errores de memoria)
X_combined = hstack([matrix_features, price_scaled]).tocsr()

# Entrenamos el KNN
model_pro = NearestNeighbors(n_neighbors=5, metric='cosine')
model_pro.fit(X_combined)

# --- EXPORTACIÓN ---
print("💾 Guardando archivos para FastAPI...")
joblib.dump(model_pro, 'modelo_knn.joblib')
joblib.dump(tfidf, 'vectorizador.joblib')
joblib.dump(scaler, 'escalador.joblib')
joblib.dump(X_combined, 'matriz_datos.joblib')

# Guardamos una versión ligera de los DataFrames para la API
df_horses.to_parquet('df_horses_api.parquet')
df_leads.to_parquet('df_leads_api.parquet')

print("🚀 ¡Todo listo! ya estan los archivos para el main.py")

💾 Guardando archivos para FastAPI...
🚀 ¡Todo listo! ya estan los archivos para el main.py
